[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 4: A Self-Supervised Learning Prototype

### Purpose and scope

This notebook introduces how a Graph Transformer could represent customers in a Vehicle Routing Problem with Backhauls (VRPB). It builds a small example, generates perturbed instances and delivery-before-pickup customer sequences, and implements a forward pass through a multi-head attention network.

The code is a **model and data-pipeline prototype**, not a complete training system. It does not implement pretext-task losses, gradient updates, fine-tuning, a routing optimizer, or route construction at inference time. Consequently, it does not produce trained embeddings, learning curves, or a measured routing cost. The three pretext tasks below describe learning objectives that could use route data; they are not trained or evaluated in this notebook.

### VRPB background and model assumptions

A depot serves two customer groups: delivery customers, whose goods are loaded at the depot and unloaded at their stops, and pickup customers, whose goods are collected and returned to the depot. In the backhaul setting used here, deliveries precede pickups on each route. A vehicle has capacity \(Q\), and customer demands are nonnegative.

Let \(D\) be the delivery customers, \(P\) the pickup customers, and \(0\) the depot. A route starts and ends at the depot. For a route \(r\), the capacity condition under the delivery-then-pickup convention is

$$
\sum_{i\in D_r} d_i \leq Q,\qquad
\sum_{j\in P_r} p_j \leq Q,
$$

where \(D_r\) and \(P_r\) are the customers of each type assigned to route \(r\), \(d_i\) is a delivery demand, and \(p_j\) is a pickup demand. The route-distance objective in a full VRPB model is to minimize the sum of travel distances over all route arcs. This notebook does not construct routes or optimize that objective; the conditions provide context for interpreting the generated customer sequences.

The example distance matrix is supplied directly. The synthetic customer coordinates are auxiliary features sampled for this prototype; they do not define or reproduce that distance matrix. The generated customer sequence places all deliveries before all pickups, but it is not guaranteed to satisfy vehicle capacity and is not an optimized VRPB solution.

### Self-supervised learning context

Self-supervised learning creates prediction targets from the structure of the available data rather than relying on manually assigned labels. For routing data, candidate pretext tasks include:

1. **Route completion:** Given a partial sequence, predict its next customer.
2. **Sequence ordering:** Given customers from a sequence, predict their relative order.
3. **Load prediction:** Given a route prefix, predict the vehicle load after servicing that prefix.

For example, if a route prefix is \(\pi_{1:t}=(\pi_1,\ldots,\pi_t)\), a route-completion target is the next customer \(\pi_{t+1}\). A load target can be defined as the remaining onboard load after the prefix, using delivery demands as load reductions and pickup demands as load increases. These objectives require meaningful route examples and explicit loss functions. The sequences generated below are random delivery-then-pickup sequences, not verified feasible or high-quality routes, so they should not be treated as expert targets.

### Graph representation and attention

Each customer is represented by a feature vector. In this implementation, the eight features are customer identifier, customer type, normalized demand, normalized coordinates, normalized distance from the coordinate origin, scaled demand, and a high-demand indicator. The Graph Transformer applies self-attention across all customer nodes: each node can form a weighted representation of every node in the input. There is no explicit edge list or distance-based attention mask in this prototype, so the implemented attention graph is fully connected over the supplied customer sequence.

For one attention head, scaled dot-product attention is

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\!\left(\frac{QK^{\mathsf T}}{\sqrt{d_h}}\right)V,
$$

where \(Q\), \(K\), and \(V\) are learned query, key, and value projections, and \(d_h\) is the dimension of a head. Multiple heads compute this operation in parallel and concatenate their outputs. The implementation uses NumPy to compute a forward pass; its weights are randomly initialized and are not updated by training.

### Workflow

1. Define the VRPB data structures and neural-network configuration.
2. Create the concrete example and inspect its demands and capacity.
3. Generate perturbed instances and delivery-before-pickup sequences.
4. Convert customer data into feature vectors.
5. Run those features through a randomly initialized Graph Transformer.
6. Interpret the output as untrained node representations, not as a routing plan or a performance result.

The model configuration specifies an input dimension of 8, a hidden dimension of 64, an output dimension of 32, 4 attention heads, and 3 layers. The hidden dimension must be divisible by the number of heads. The input projection maps the eight-dimensional features to the hidden dimension before the transformer layers, so residual connections operate on vectors of matching size.


In [ ]:
# NumPy supplies the array operations used by the data pipeline and forward pass.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Any, List, Tuple
import random

# Keep plots consistent if visual diagnostics are added to the prototype.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for the VRPB self-supervised learning prototype.")

In [ ]:
@dataclass
class VRPBInstance:
    """Store customer demands, vehicle capacity, and pairwise distances."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self) -> None:
        if self.num_deliveries < 0 or self.num_pickups < 0:
            raise ValueError("Customer counts must be nonnegative.")
        if len(self.delivery_demands) != self.num_deliveries:
            raise ValueError("The number of delivery demands must match num_deliveries.")
        if len(self.pickup_demands) != self.num_pickups:
            raise ValueError("The number of pickup demands must match num_pickups.")
        if self.capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")
        if any(demand < 0 for demand in self.delivery_demands + self.pickup_demands):
            raise ValueError("Customer demands must be nonnegative.")

        self.num_customers = self.num_deliveries + self.num_pickups
        expected_shape = (self.num_customers + 1, self.num_customers + 1)
        if self.distance_matrix.shape != expected_shape:
            raise ValueError(
                f"Distance matrix must have shape {expected_shape}; "
                f"received {self.distance_matrix.shape}."
            )

        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class CustomerNode:
    """Represent one customer and its auxiliary spatial features."""

    customer_id: int
    customer_type: str
    demand: float
    x_coord: float
    y_coord: float
    features: List[float] = field(default_factory=list)


@dataclass
class GraphTransformerConfig:
    """Hold dimensions and nominal settings for the Graph Transformer."""

    input_dim: int = 8
    hidden_dim: int = 64
    output_dim: int = 32
    num_heads: int = 4
    num_layers: int = 3
    dropout_rate: float = 0.1
    learning_rate: float = 0.001


@dataclass
class PretextTask:
    """Describe a pretext task and its data, target, and optional prediction."""

    task_name: str
    input_data: Any
    target_data: Any
    prediction: Any = None
    loss: float = 0.0


print("VRPB data structures and model configuration are defined.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the stated example instance with its given distance matrix."""

    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    # Rows and columns are ordered as depot, deliveries, then pickups.
    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ],
        dtype=float,
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


# Inspect the instance inputs; these totals alone do not define a route plan.
instance = create_concrete_example()
print(
    f"Created an instance with {instance.num_deliveries} deliveries "
    f"and {instance.num_pickups} pickups."
)
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
class VRPBDataGenerator:
    """Generate perturbed instances and candidate customer sequences."""

    def __init__(self, base_instance: VRPBInstance):
        self.base_instance = base_instance

    def generate_customer_nodes(
        self, instance: VRPBInstance
    ) -> List[CustomerNode]:
        """Create customer nodes with reproducible auxiliary coordinates."""
        nodes = []
        np.random.seed(42)

        for i, customer_id in enumerate(instance.delivery_indices):
            x_coord = np.random.uniform(0, 20)
            y_coord = np.random.uniform(0, 20)
            nodes.append(
                CustomerNode(
                    customer_id=customer_id,
                    customer_type="delivery",
                    demand=instance.delivery_demands[i],
                    x_coord=x_coord,
                    y_coord=y_coord,
                )
            )

        for i, customer_id in enumerate(instance.pickup_indices):
            x_coord = np.random.uniform(0, 20)
            y_coord = np.random.uniform(0, 20)
            nodes.append(
                CustomerNode(
                    customer_id=customer_id,
                    customer_type="pickup",
                    demand=instance.pickup_demands[i],
                    x_coord=x_coord,
                    y_coord=y_coord,
                )
            )

        return nodes

    def extract_node_features(
        self, node: CustomerNode, instance: VRPBInstance
    ) -> List[float]:
        """Build the eight-dimensional input vector used by the model."""
        features = [
            float(node.customer_id),
            1.0 if node.customer_type == "delivery" else 0.0,
            node.demand / instance.capacity,
            node.x_coord / 20.0,
            node.y_coord / 20.0,
            np.sqrt(node.x_coord**2 + node.y_coord**2) / 20.0,
            node.demand / 10.0,
            1.0 if node.demand > instance.capacity * 0.5 else 0.0,
        ]
        node.features = features
        return features

    def generate_training_instances(
        self, num_instances: int = 100
    ) -> List[Tuple[VRPBInstance, List[int]]]:
        """Generate perturbed instances with random delivery-first sequences.

        The returned sequence is a candidate ordering, not a verified feasible
        route or an optimal solution. It is not used as a training target here.
        """
        if num_instances < 0:
            raise ValueError("num_instances must be nonnegative.")

        training_data = []

        for i in range(num_instances):
            # A per-instance seed makes demand and distance perturbations repeatable.
            np.random.seed(i)
            delivery_demands = [
                max(1, demand + np.random.randint(-1, 2))
                for demand in self.base_instance.delivery_demands
            ]
            pickup_demands = [
                max(1, demand + np.random.randint(-1, 2))
                for demand in self.base_instance.pickup_demands
            ]

            noise_matrix = np.random.normal(
                0,
                0.5,
                self.base_instance.distance_matrix.shape,
            )
            distance_matrix = np.clip(
                self.base_instance.distance_matrix + noise_matrix,
                1,
                None,
            )
            distance_matrix = (distance_matrix + distance_matrix.T) / 2
            np.fill_diagonal(distance_matrix, 0)

            perturbed_instance = VRPBInstance(
                num_deliveries=self.base_instance.num_deliveries,
                num_pickups=self.base_instance.num_pickups,
                delivery_demands=delivery_demands,
                pickup_demands=pickup_demands,
                capacity=self.base_instance.capacity,
                distance_matrix=distance_matrix,
            )

            deliveries = perturbed_instance.delivery_indices.copy()
            pickups = perturbed_instance.pickup_indices.copy()
            sequence_rng = random.Random(i)
            sequence_rng.shuffle(deliveries)
            sequence_rng.shuffle(pickups)
            candidate_sequence = deliveries + pickups
            training_data.append((perturbed_instance, candidate_sequence))

        return training_data


print("The synthetic VRPB data generator is defined.")

### Reading the data-generation code

`generate_training_instances` perturbs the customer demands and the supplied distance matrix, then creates a customer ordering with deliveries before pickups. The per-instance NumPy seed controls the numeric perturbations, and a local Python random generator controls the sequence shuffles. This makes the generated examples reproducible without relying on the global Python random state.

The name `training_instances` refers to the intended role of the data in a future learning pipeline. In this notebook, no loss is computed from these sequences and no model parameters are trained. In particular, delivery-first ordering enforces precedence in the sequence but does not ensure that all demands fit on one vehicle. The feature coordinates are sampled independently of the supplied distance matrix, so they are auxiliary node attributes rather than a geometric reconstruction of travel distances.


In [ ]:
class GraphTransformerLayer:
    """Apply multi-head self-attention and a feed-forward transformation."""

    def __init__(self, input_dim: int, output_dim: int, num_heads: int):
        if output_dim % num_heads != 0:
            raise ValueError("output_dim must be divisible by num_heads.")

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = output_dim // num_heads

        # Fixed random weights support a forward-pass demonstration, not training.
        np.random.seed(42)
        self.query_weights = np.random.randn(input_dim, output_dim) * 0.1
        self.key_weights = np.random.randn(input_dim, output_dim) * 0.1
        self.value_weights = np.random.randn(input_dim, output_dim) * 0.1
        self.output_weights = np.random.randn(output_dim, output_dim) * 0.1

        self.gamma = np.ones(output_dim)
        self.beta = np.zeros(output_dim)

    def multi_head_attention(
        self,
        query: np.ndarray,
        key: np.ndarray,
        value: np.ndarray,
    ) -> np.ndarray:
        """Compute scaled dot-product attention independently per head."""
        batch_size, seq_len, _ = query.shape

        projected_query = np.dot(query, self.query_weights)
        projected_key = np.dot(key, self.key_weights)
        projected_value = np.dot(value, self.value_weights)

        projected_query = projected_query.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(0, 2, 1, 3)
        projected_key = projected_key.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(0, 2, 1, 3)
        projected_value = projected_value.reshape(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(0, 2, 1, 3)

        scores = np.matmul(
            projected_query,
            projected_key.transpose(0, 1, 3, 2),
        ) / np.sqrt(self.head_dim)
        attention_weights = self.softmax(scores, axis=-1)
        context = np.matmul(attention_weights, projected_value)

        # Concatenation returns one output vector per input customer.
        context = context.transpose(0, 2, 1, 3).reshape(
            batch_size, seq_len, self.output_dim
        )
        return context

    def softmax(self, x: np.ndarray, axis: int = -1) -> np.ndarray:
        """Normalize scores into stable probability weights along one axis."""
        shifted_x = x - np.max(x, axis=axis, keepdims=True)
        exp_x = np.exp(shifted_x)
        return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

    def layer_norm(self, x: np.ndarray) -> np.ndarray:
        """Normalize each node representation across its feature dimension."""
        mean = np.mean(x, axis=-1, keepdims=True)
        variance = np.mean((x - mean) ** 2, axis=-1, keepdims=True)
        normalized = (x - mean) / np.sqrt(variance + 1e-6)
        return self.gamma * normalized + self.beta

    def gelu(self, x: np.ndarray) -> np.ndarray:
        """Apply the Gaussian Error Linear Unit activation approximation."""
        return 0.5 * x * (
            1
            + np.tanh(
                np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)
            )
        )

    def forward(self, x: np.ndarray) -> np.ndarray:
        """Apply attention, residual normalization, and a feed-forward step."""
        attention_output = self.multi_head_attention(x, x, x)
        x = self.layer_norm(x + attention_output)

        feed_forward_output = np.dot(x, self.output_weights)
        feed_forward_output = self.gelu(feed_forward_output)
        return self.layer_norm(x + feed_forward_output)


class GraphTransformer:
    """Map customer feature vectors to untrained node representations."""

    def __init__(self, config: GraphTransformerConfig):
        if config.num_layers < 1:
            raise ValueError("num_layers must be at least 1.")
        if config.hidden_dim % config.num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads.")

        self.config = config
        np.random.seed(42)
        self.input_projection = (
            np.random.randn(config.input_dim, config.hidden_dim) * 0.1
        )

        self.layers = [
            GraphTransformerLayer(
                input_dim=config.hidden_dim,
                output_dim=config.hidden_dim,
                num_heads=config.num_heads,
            )
            for _ in range(config.num_layers)
        ]

        self.output_projection = (
            np.random.randn(config.hidden_dim, config.output_dim) * 0.1
        )

    def forward(self, node_features: np.ndarray) -> np.ndarray:
        """Return one output embedding for each customer in the input."""
        if node_features.ndim != 3:
            raise ValueError(
                "node_features must have shape (batch, customers, features)."
            )
        if node_features.shape[-1] != self.config.input_dim:
            raise ValueError(
                f"Expected {self.config.input_dim} input features, "
                f"received {node_features.shape[-1]}."
            )

        x = np.dot(node_features, self.input_projection)
        for layer in self.layers:
            x = layer.forward(x)
        return np.dot(x, self.output_projection)


print("The Graph Transformer forward-pass prototype is defined.")

### Interpreting the network implementation

The input projection converts each eight-feature customer vector into a 64-dimensional hidden representation. Each transformer layer then preserves that hidden dimension so the residual additions are defined. With 4 heads, each head uses \(d_h=64/4\) features. The output projection maps each final hidden representation to 32 dimensions.

The forward pass expects an array of shape `(batch, customers, features)`. A single instance is therefore represented with a batch dimension of size one. The output has shape `(batch, customers, output_dim)`: it contains a vector for each input customer, not a route, a next-customer decision, or an objective value.

The implementation uses fixed random weights. The configuration fields `learning_rate` and `dropout_rate` are retained as model settings, but they have no effect in this forward-only prototype. No loss or optimizer is implemented, so there is no basis here for reporting training progress or learned routing behavior.


In [ ]:
# Build customer features and pass them through the initialized model.
data_generator = VRPBDataGenerator(instance)
customer_nodes = data_generator.generate_customer_nodes(instance)
node_features = np.array(
    [
        data_generator.extract_node_features(node, instance)
        for node in customer_nodes
    ],
    dtype=float,
)

config = GraphTransformerConfig()
model = GraphTransformer(config)
batched_node_features = node_features[np.newaxis, :, :]
node_embeddings = model.forward(batched_node_features)

assert node_features.shape == (instance.num_customers, config.input_dim)
assert node_embeddings.shape == (
    1,
    instance.num_customers,
    config.output_dim,
)
print(f"Customer feature matrix shape: {node_features.shape}")
print(f"Untrained embedding tensor shape: {node_embeddings.shape}")

In [ ]:
# Generate a small reproducible sample to inspect the data structures.
training_data = data_generator.generate_training_instances(num_instances=5)
assert len(training_data) == 5

for generated_instance, candidate_sequence in training_data:
    assert len(candidate_sequence) == generated_instance.num_customers
    assert set(candidate_sequence) == set(generated_instance.all_customer_indices)
    delivery_positions = [
        candidate_sequence.index(customer_id)
        for customer_id in generated_instance.delivery_indices
    ]
    pickup_positions = [
        candidate_sequence.index(customer_id)
        for customer_id in generated_instance.pickup_indices
    ]
    if delivery_positions and pickup_positions:
        assert max(delivery_positions) < min(pickup_positions)

print("Generated candidate sequences contain every customer and place deliveries first.")
print("Capacity feasibility and route quality are not assessed by these checks.")

## Summary and conclusions

### What this prototype demonstrates

- A VRPB instance can be stored with separate delivery and pickup demands, a vehicle capacity, and a distance matrix.
- Synthetic instances can be generated by perturbing demands and distances while retaining the delivery-before-pickup ordering convention in candidate sequences.
- Customer features can be passed through a multi-head self-attention architecture to produce one vector per customer.
- Shape and ordering assertions can check basic properties of the generated data and model output.

These demonstrations establish that the data pipeline and forward-pass code are connected. They do **not** establish that the model has learned useful representations, that the candidate sequences are feasible routes, or that a neural model can solve the VRPB instance.

### Pretext-task status

Route completion, sequence ordering, and load prediction are candidate self-supervised objectives described in the modeling context. The notebook defines a `PretextTask` data structure but does not create task examples, calculate task losses, or update model weights. Since the generated sequences are random and are not checked for capacity feasibility or route quality, treating them as high-quality routing targets would not be justified.

A complete self-supervised experiment would need task-specific targets and losses, a training procedure that updates the network parameters, and evaluation on clearly defined routing criteria. None of those results can be inferred from a forward pass with randomly initialized weights.

### Method assessment

**What the prototype is useful for:**

- Inspecting how VRPB instance data and customer features can be organized.
- Following the dimensional flow from customer features through attention layers to node representations.
- Providing a starting point for implementing the three stated pretext tasks within the same Graph Transformer approach.

**Limitations of the current implementation:**

- The network weights are initialized randomly and remain fixed; there is no training or learning-curve result.
- The candidate customer sequences are not guaranteed to satisfy vehicle capacity and are not optimized using the distance matrix.
- The auxiliary coordinates are generated independently of the supplied travel distances.
- Attention is applied to all input customers without an explicit graph-edge representation or distance mask.
- The forward-pass outputs are not converted into a feasible route or evaluated as routing solutions.

### Comparison with classical optimization approaches

| Aspect | Classical optimization model | This self-attention prototype |
|---|---|---|
| Primary role | Formulate and optimize routing decisions | Compute node representations from features |
| Feasibility | Expressed through explicit route constraints | Not enforced by the forward pass |
| Objective value | Defined and evaluated by the routing model | Not computed here |
| Learning | Not required by a classical formulation | No training is implemented here |
| Interpretation | Decisions and constraints can be inspected | Attention-based representations require further analysis |

### Scope of conclusions

This notebook is an architectural and data-pipeline illustration of a possible self-supervised VRPB approach. It does not provide evidence of competitive performance, generalization to new instances, learned attention patterns, or a reduction in routing cost. Such conclusions require a completed training and evaluation procedure using feasible, well-defined routing data.
